https://huggingface.co/Qwen/Qwen2-Audio-7B-Instruct

In [1]:
from io import BytesIO
from urllib.request import urlopen
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor, AutoTokenizer

model_path = "/home/xwchen/data/models/r1-aqa"


In [2]:
processor = AutoProcessor.from_pretrained(model_path)

Some kwargs in processor config are unused and will not have any effect: chat_template, audio_eos_token, audio_bos_token, audio_token. 


In [3]:
model = Qwen2AudioForConditionalGeneration.from_pretrained(model_path, device_map="auto")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
conversation = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": "/home/xwchen/data/VGGSound/audio/-HxQ9AoyRmY_000030.wav"},
        {"type": "text", "text": "What is shown in the audio? Choose an answer from ['A flock of chickens', 'frog', 'gibbon', 'duck']. Let's think step by step and output the final answer in <answer></answer>."},
    ]},
]
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios = []
for message in conversation:
    if isinstance(message["content"], list):
        for ele in message["content"]:
            if ele["type"] == "audio":
                audios.append(librosa.load(
                    ele['audio_url'], 
                    sr=processor.feature_extractor.sampling_rate)[0]
                )

inputs = processor(text=text, audios=audios, return_tensors="pt", padding=True)
inputs.input_ids = inputs.input_ids.to("cuda")

generate_ids = model.generate(**inputs, max_length=512)
generate_ids = generate_ids[:, inputs.input_ids.size(1):]

response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
response

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
/home/xwchen/miniconda3/envs/verl/lib/python3.11/site-packages/transformers/generation/utils.py:2105: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


'<answer>A flock of chickens</answer>'

In [ ]:
import datasets
from torchdata.stateful_dataloader import StatefulDataLoader
dataset = datasets.load_from_disk("/home/xwchen/data/aqa")
train_dataset = dataset['train']
test_dataset = dataset['test']

In [ ]:
for train_data in train_dataset:
    conversation = [
        {"role": "user", "content": [
            {"type": "audio", "audio_url": train_data['audio_path']},
            {"type": "text", "text": train_data["question_text"].replace("video", "audio") + " Choose an answer from "+ str(train_data["multi_choice"]) + ". Let's think step by step and output the final answer in <answer></answer>."},
        ]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios = []
    for message in conversation:
        if isinstance(message["content"], list):
            for ele in message["content"]:
                if ele["type"] == "audio":
                    audios.append(librosa.load(
                        ele['audio_url'], 
                        sr=processor.feature_extractor.sampling_rate)[0]
                    )

    inputs = processor(text=text, audios=audios, return_tensors="pt", padding=True)
    inputs.input_ids = inputs.input_ids.to("cuda")

    generate_ids = model.generate(**inputs, max_length=512)
    generate_ids = generate_ids[:, inputs.input_ids.size(1):]

    response = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    if train_data["multi_choice"][train_data['answer']] in response:
        print(True)
    else:
        print(False)

In [ ]:
train_dataloader = StatefulDataLoader(
    dataset=train_dataset, batch_size=128, num_workers=8, drop_last=True
)

In [30]:
del processor, model